In [ ]:
!pip install -q transformers accelerate bitsandbytes pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Unconstrained Prompt

In [ ]:
import os
import json
import torch
from PIL import Image
from transformers import pipeline, BitsAndBytesConfig

folder_path = '/content/drive/MyDrive/allRotatedPhotos'
output_json = '/content/drive/MyDrive/analys_masks.json'

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading InternVL3-8B-hf...")
pipe = pipeline(
    "image-text-to-text",
    model="OpenGVLab/InternVL3-8B-hf",
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto"
)

extraction_prompt = """
Extract all text from this turbocharger nameplate.
Maintain the line structure.
If you see serial numbers or part numbers, ensure they are exact.
CRITICAL: The text on this nameplate consists ONLY of Latin letters, numbers, hyphens (-), and slashes (/). You are strictly forbidden from outputting any other symbols like cyrillic letters or punctuation.
Return ONLY the extracted text, no introductory sentences.
"""

supported_extensions = ('.jpg', '.jpeg', '.png', '.webp')
results = {}

if os.path.exists(output_json):
    os.remove(output_json)
    print("🗑️ Deleted old JSON data. Starting completely fresh!\n")

total_session_input_tokens = 0
total_session_output_tokens = 0

if not os.path.exists(folder_path):
    print(f"Error: The folder '{folder_path}' does not exist.")
else:
    image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(supported_extensions)]
    print(f"Found {len(image_files)} images total. Starting extraction...")

    for i, filename in enumerate(image_files, start=1):
        file_path = os.path.join(folder_path, filename)
        print(f"\n[{i}/{len(image_files)}] Processing: {filename}")

        try:
            image = Image.open(file_path).convert("RGB")

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": extraction_prompt}
                    ]
                }
            ]

            outputs = pipe(
                messages,
                max_new_tokens=512,
                return_full_text=False
            )

            result_text = outputs[0]["generated_text"]

            input_tokens = len(pipe.tokenizer.encode(extraction_prompt))
            output_tokens = len(pipe.tokenizer.encode(result_text))

            total_session_input_tokens += input_tokens
            total_session_output_tokens += output_tokens

            results[filename] = result_text.strip()

            os.makedirs(os.path.dirname(output_json), exist_ok=True)
            with open(output_json, 'w', encoding='utf-8') as f:
                json.dump(results, f, indent=4, ensure_ascii=False)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

        finally:
            torch.cuda.empty_cache()

    print(f"\n✅ Finished processing! All data is saved in: {output_json}\n")

    grand_total_tokens = total_session_input_tokens + total_session_output_tokens

    summary_text = (
        f"--- Token Usage Summary ---\n"
        f"Total Images Processed: {len(image_files)}\n"
        f"Total Input Text Tokens: {total_session_input_tokens}\n"
        f"Total Output Tokens: {total_session_output_tokens}\n"
        f"Grand Total Tokens: {grand_total_tokens}\n"
        f"---------------------------\n"
    )
    print(summary_text)

Constrained Prompt

In [ ]:
import os
import json
import torch
from PIL import Image
from transformers import pipeline, BitsAndBytesConfig

folder_path = '/content/drive/MyDrive/allRotatedPhotos'
output_json = '/content/drive/MyDrive/analys_masks.json'

# 2. Configure 4-bit Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

print("Loading InternVL3-8B-hf...")
pipe = pipeline(
    "image-text-to-text",
    model="OpenGVLab/InternVL3-8B-hf",
    model_kwargs={"quantization_config": quantization_config},
    device_map="auto"
)

extraction_prompt = """
Analyze this Garrett turbocharger nameplate and extract the information into a strict JSON format.

Follow these two rules:
1. "raw_text": Extract all visible text from the nameplate. Keep it exact, with no introductory text, descriptions, or added formatting. The text on this nameplate consists ONLY of Latin letters, numbers, hyphens (-), and slashes (/). You are strictly forbidden from outputting any other symbols like cyrillic letters or punctuation.

2. "part_number": Extract ONLY the Garrett Part Number. It must follow the format of 6 digits, a hyphen (-), and 1 to 4 digits (e.g., 758219-3).
It MUST start with the digit 4, 7, or 8. If you cannot find a matching sequence, return null.

CRITICAL: Return ONLY valid JSON. Do not include markdown backticks (```) or any other conversational text.
Expected format:
{
  "raw_text": "all the text from the plate goes here...",
  "part_number": "XXXXXX-X..."
}
"""

supported_extensions = ('.jpg', '.jpeg', '.png', '.webp')
final_results_list = []

if os.path.exists(output_json):
    os.remove(output_json)
    print("🗑️ Deleted old JSON data. Starting fresh!\n")

total_session_input_tokens = 0
total_session_output_tokens = 0

if not os.path.exists(folder_path):
    print(f"Error: The folder '{folder_path}' does not exist.")
else:
    image_files = sorted([f for f in os.listdir(folder_path) if f.lower().endswith(supported_extensions)])
    print(f"Found {len(image_files)} images total. Starting extraction...")

    for i, filename in enumerate(image_files, start=1):
        file_path = os.path.join(folder_path, filename)
        print(f"\n[{i}/{len(image_files)}] Processing: {filename}")

        try:
            image = Image.open(file_path).convert("RGB")

            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image},
                        {"type": "text", "text": extraction_prompt}
                    ]
                }
            ]

            outputs = pipe(messages, max_new_tokens=512, return_full_text=False)
            result_raw = outputs[0]["generated_text"].strip()

            try:
                clean_json = result_raw.replace("```json", "").replace("```", "").strip()
                extracted_data = json.loads(clean_json)
            except Exception:
                extracted_data = {"raw_text": result_raw, "part_number": None}

            entry = {
                "image_name": filename,
                "part_number": extracted_data.get("part_number"),
                "raw_text": extracted_data.get("raw_text")
            }
            final_results_list.append(entry)

            total_session_input_tokens += len(pipe.tokenizer.encode(extraction_prompt))
            total_session_output_tokens += len(pipe.tokenizer.encode(result_raw))

            os.makedirs(os.path.dirname(output_json), exist_ok=True)
            with open(output_json, 'w', encoding='utf-8') as f:
                json.dump(final_results_list, f, indent=4, ensure_ascii=False)

        except Exception as e:
            print(f"Error processing {filename}: {e}")

        finally:
            torch.cuda.empty_cache()

    print(f"\n✅ Finished processing! Saved to: {output_json}")

    # --- Print Token Usage Summary ---
    grand_total_tokens = total_session_input_tokens + total_session_output_tokens

    summary_text = (
        f"\n--- Token Usage Summary ---\n"
        f"Total Images Processed: {len(image_files)}\n"
        f"Total Input Tokens:     {total_session_input_tokens}\n"
        f"Total Output Tokens:    {total_session_output_tokens}\n"
        f"Grand Total Tokens:     {grand_total_tokens}\n"
    )
    print(summary_text)